In [ ]:
!apt-get install tesseract-ocr
!pip install pytesseract Pillow SpeechRecognition
!pip install openai-whisper
pip install openai torch faiss-cpu numpy transformers whisper pillow gradio

In [ ]:
import os
import shutil
import pytesseract
from PIL import Image
import speech_recognition as sr
from sentence_transformers import SentenceTransformer, util

# Setup folders
DOCUMENTS_FOLDER = '/content/documents'
if not os.path.exists(DOCUMENTS_FOLDER):
    os.makedirs(DOCUMENTS_FOLDER)

# Initialize database and model
database = []  # Each item: {'type': 'text'/'image'/'audio', 'content': <text>, 'embedding': <vector>}
model = SentenceTransformer('all-MiniLM-L6-v2')

def add_text():
    text = input("Enter text: ")
    embedding = model.encode(text, convert_to_tensor=True)
    database.append({'type': 'text', 'content': text, 'embedding': embedding})
    print("Document added successfully.")

def add_image():
    image_path = input("Enter image file path: ")
    try:
        # Save image file
        shutil.copy(image_path, DOCUMENTS_FOLDER)
        # Extract text from image
        extracted_text = pytesseract.image_to_string(Image.open(image_path))
        embedding = model.encode(extracted_text, convert_to_tensor=True)
        database.append({'type': 'image', 'content': extracted_text, 'embedding': embedding})
        print("Image added and text extracted successfully.")
    except Exception as e:
        print(f"Failed to add image: {e}")

def add_audio():
    audio_path = input("Enter audio file path: ")
    try:
        # Save audio file
        shutil.copy(audio_path, DOCUMENTS_FOLDER)
        # Extract text from audio
        recognizer = sr.Recognizer()
        with sr.AudioFile(audio_path) as source:
            audio = recognizer.record(source)
        extracted_text = recognizer.recognize_google(audio)
        embedding = model.encode(extracted_text, convert_to_tensor=True)
        database.append({'type': 'audio', 'content': extracted_text, 'embedding': embedding})
        print("Audio added and text extracted successfully.")
    except Exception as e:
        print(f"Failed to add audio: {e}")

def query():
    query_text = input("Enter your query text: ")
    query_embedding = model.encode(query_text, convert_to_tensor=True)

    best_match = None
    best_score = -1

    for doc in database:
        score = util.cos_sim(query_embedding, doc['embedding']).item()
        if score > best_score:
            best_score = score
            best_match = doc

    if best_match and best_score > 0.5:  # Optional: Threshold to ensure meaningful matches
        print(f"Answer (from {best_match['type']}): {best_match['content']}")
    else:
        print("No relevant document found.")

def main():
    while True:
        action = input("Choose action (add_text / add_image / add_audio / query / exit): ")
        if action == 'add_text':
            add_text()
        elif action == 'add_image':
            add_image()
        elif action == 'add_audio':
            add_audio()
        elif action == 'query':
            query()
        elif action == 'exit':
            print("Exiting...")
            break
        else:
            print("Invalid action. Please try again.")

if __name__ == "__main__":
    main()
